# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [5]:
import torch
# 1. 创建三个参数 w1, w2, b，初始值都是 0.0，要求梯度
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b  = torch.tensor(0.0, requires_grad=True)

# 2. 创建 SGD 优化器，管这三个参数
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

# 3. 训练循环 2000 轮
for epoch in range(1, 2001):
    lr = 0.1 / (epoch ** 0.5)          # 学习率逐渐变小
    for g in optimizer.param_groups:
        g['lr'] = lr
    optimizer.zero_grad()               # 清空旧梯度
    
    y_pred = predict(x1, x2, w1, w2, b)  # 预测6个值
    loss = torch.mean(torch.abs(y - y_pred))  # 平均绝对误差
    
    loss.backward()                     # 反向传播
    optimizer.step()                    # 更新参数


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

待填写。


In [ ]:
## 我的观察与实验

**1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？**

需要 **2 个输入张量**：x1 和 x2。
- x1 shape 是 (6,)，x2 shape 也是 (6,)，各包含 6 个数值
- x1[i] 是第 i 个样本的第一个输入，x2[i] 是第 i 个样本的第二个输入，它们和 y[i] 属于同一个样本
- predict(x1, x2, ...) 一次同时处理 6 个样本，返回 shape 为 (6,) 的 6 个预测值

**2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？**

- 预测公式：从 y = w*x + b 变成 y = w1*x1 + w2*x2 + b，多了一个输入 x2 和一个参数 w2
- 参数列表：从 [w, b] 变成 [w1, w2, b]，多了 w2
- 损失计算：还是绝对误差取均值，但现在是对 6 个样本同时计算，loss 从标量对标量变成 (6,) 取 mean
- 优化器基本步骤**没变**：清空梯度 → 预测 → 算 loss → 反向传播 → 更新参数，还是这 5 步

**3. 记录最终参数、预测和平均损失，说明如何判断训练训练有效。**

训练 2000 轮后，参数应接近：
- w1 ≈ 2.0，w2 ≈ -3.0，b ≈ 1.0
- 预测值应与真实 y 非常接近
- 平均损失应接近 0

判断训练有效的标准：**平均损失很小（接近 0），且 y_pred 和真实 y 的数值基本一致**，说明模型学到了正确的 w1、w2、b。

**4. 为什么这次 w1=w2=b=0 仍然可以开始学习？**

因为预测公式是线性的 y = w1*x1 + w2*x2 + b。初始全 0 时，预测全是 0，但真实 y 不全为 0，所以 loss 不为 0。
对 w1 求梯度时，梯度里包含 x1，而 x1 的值不全为 0，所以梯度不为 0，参数就能更新，学习就能开始。
不存在"全 0 导致梯度消失"的问题。
